# AksaraDetect - Full Project di Google Colab

**Sebelum mulai:**
1. Klik **Runtime** -> **Change runtime type** -> pilih **T4 GPU** -> Save
2. Ganti `GANTI_API_KEY_KAMU` di Cell 2 dengan API key Roboflow kamu
3. Klik **Runtime** -> **Run all**
4. Tunggu ~15 menit, lalu klik URL yang muncul di Cell terakhir

**Cara dapat API Key:** buka https://app.roboflow.com -> klik foto profil -> Settings -> Roboflow API

In [ ]:
# Cell 1 - Install semua dependensi
!nvidia-smi
!pip install ultralytics roboflow streamlit pyngrok PyYAML matplotlib seaborn pandas Pillow scikit-learn -q
print("Install selesai!")

In [ ]:
# Cell 2 - Download dataset dari Roboflow
from roboflow import Roboflow
import yaml, os

API_KEY = "GANTI_API_KEY_KAMU"  # <-- GANTI INI
# Cara dapat: buka app.roboflow.com -> Settings -> Roboflow API

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("novalrizkiansyah-ymail-com").project("aksara-ulu-rejang")
dataset = project.version(4).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
base = os.path.dirname(DATA_YAML)

with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

data["train"] = base + "/train/images"
data["val"]   = base + "/valid/images"
data["test"]  = base + "/test/images"

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f)

print(f"Dataset siap! Kelas: {data['nc']}")

In [ ]:
# Cell 3 - Clone project dari GitHub
!git clone https://github.com/ReffkiAndreaPratama/aksara-detect.git /content/app
print("Project berhasil di-clone!")

In [ ]:
# Cell 4 - Training YOLOv8 dengan GPU (~10-15 menit)
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data     = DATA_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 32,
    device   = 0,
    project  = "/content/runs",
    name     = "aksara_ulu",
    exist_ok = True,
)
print("Training selesai!")
print("mAP50:", results.results_dict.get("metrics/mAP50(B)", "N/A"))

In [ ]:
# Cell 5 - Setup model, label map, dan config
import os, json, shutil, yaml

for d in ["/content/app/models", "/content/app/results", "/content/app/logs"]:
    os.makedirs(d, exist_ok=True)

shutil.copy("/content/runs/aksara_ulu/weights/best.pt", "/content/app/models/best.pt")

with open(DATA_YAML) as f:
    d = yaml.safe_load(f)
label_map = {str(i): name for i, name in enumerate(d["names"])}
with open("/content/app/models/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

config_lines = [
    "import os\n",
    'BASE_DIR        = "/content/app"\n',
    f'DATA_YAML       = "{DATA_YAML}"\n',
    'MODEL_DIR       = "/content/app/models"\n',
    'RESULT_DIR      = "/content/app/results"\n',
    'LOG_DIR         = "/content/app/logs"\n',
    'RUNS_DIR        = "/content/app/runs"\n',
    'MODEL_BEST_PATH = "/content/app/models/best.pt"\n',
    'MODEL_LAST_PATH = "/content/app/models/last.pt"\n',
    'LABEL_MAP_PATH  = "/content/app/models/label_map.json"\n',
    'METRICS_PATH    = "/content/app/models/metrics.json"\n',
    'PREDICT_LOG     = "/content/app/logs/prediction_log.json"\n',
    "CONF_THRESHOLD  = 0.25\n",
    "IOU_THRESHOLD   = 0.45\n",
    'APP_NAME        = "AksaraDetect"\n',
    'APP_VERSION     = "1.0.0"\n',
    'APP_TAGLINE     = "Aksara Ulu Rejang - YOLOv8"\n',
]
with open("/content/app/src/config.py", "w") as f:
    f.writelines(config_lines)

print(f"Setup selesai! {len(label_map)} kelas siap.")

In [ ]:
# Cell 6 - Jalankan GUI Streamlit + buat URL publik
import subprocess, time, sys
from pyngrok import ngrok

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run",
     "/content/app/src/app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    cwd="/content/app/src"
)

time.sleep(10)

tunnel = ngrok.connect(8501)
print("=" * 55)
print("AKSARADETECT SIAP!")
print("Buka URL ini di browser:")
print("  " + tunnel.public_url)
print("=" * 55)
print("URL aktif selama session Colab berjalan")